# LAB: Agents that Speak MCP Fluently

**Ayhan Demir** · Ironhack AI Consulting & Integration Bootcamp · Module 4, Week 4

## Objective

In this lab we connect **two very different servers**:

- 🌐 a **Fetch server** that runs **on our own laptop** and reads public web pages
- 🐙 a **GitHub server** that runs **on GitHub's computers** and reads code repositories

The goal is not just a working agent. The goal is a **provably grounded** agent: for every answer, we can point to the **exact tool call and the exact text** that backs it up.

## Use case: agent framework shortlist for a client RFP

A client asks which agent framework to shortlist. A consultant needs **facts, not opinions**: the license and latest release of each framework (from **GitHub**) and what each framework says it is for (from its **documentation page**, read with **Fetch**). We compare **LangGraph** and the **OpenAI Agents SDK**.

## 🗺️ Steps we will follow

| # | Step | What happens |
|---|------|--------------|
| 1 | **Setup and installation** | Install packages, load the OpenAI key and the GitHub token, check `uvx` |
| 2 | **Connect to MCP servers** | One client per server: Fetch (**stdio**) and GitHub (**HTTP** + token). List tools and resources |
| 3 | **Load MCP tools into LangChain** | Load the tools and check they are accessible |
| 4 | **Create an agent with MCP tools** | Ask one question per server and read every tool call behind the answer |
| 5 | **Access MCP resources** | Give an agent read-only background from the LangChain Docs server |
| 6 | **Build a complete MCP-enabled agent** | Two clients, one agent, one RFP question that needs both servers |
| 7 | Optional: MCP vs direct API | Not done (optional) |
| 8 | Optional: practical example | Not done (optional) |
| ✅ | **`lab_proof.md`** | The entry point for the reviewer |

**Sources**
- Documentation pages: `https://docs.langchain.com/oss/python/langgraph/overview` and `https://openai.github.io/openai-agents-python/`
- GitHub repositories: `langchain-ai/langgraph` and `openai/openai-agents-python`
- MCP resource: the LangChain skill guide from `https://docs.langchain.com/mcp`

> Run `mcp_setup_mac.ipynb` first if `uvx` or the GitHub token is missing.

---
## Step 1: Setup and Installation

1. `.env` file next to this notebook (never committed):

| Server | Add to `.env` |
|---|---|
| All | `OPENAI_API_KEY=sk-...` |
| Fetch | Nothing |
| LangChain Docs | Nothing |
| **GitHub** | `GITHUB_PERSONAL_ACCESS_TOKEN=github_pat_...` (fine-grained, *Public repositories (read-only)*) |

2. The Fetch server is a **Python** server, started with `uvx` (from `uv`). `mcp_setup_mac.ipynb` installs it.

In [1]:
%pip install -qU langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv "pydantic>=2,<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import shutil
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it to your .env file.")

GITHUB_TOKEN = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")   # only needed for the GitHub server
# uvx starts the Fetch server: from Homebrew, or from this environment (pip install uv)
UVX = shutil.which("uvx") or os.path.join(os.path.dirname(sys.executable), "uvx")

print("OpenAI key:   loaded")
print("uvx:         ", "found" if os.path.exists(UVX) else "NOT found: run mcp_setup_mac.ipynb")
print("GitHub token:", "loaded" if GITHUB_TOKEN else "missing: needed for the GitHub server")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

OpenAI key:   loaded
uvx:          found
GitHub token: loaded


In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

async def log_tool_calls(request, handler):
    # BEFORE: which server, which tool, which inputs
    print(f"[LOG] {request.server_name} -> {request.name}  {request.args}")
    result = await handler(request)   # the real tool runs on the server
    # AFTER: what the server sent back
    text = " ".join(getattr(part, "text", str(part)) for part in result.content)
    print(f"[LOG]    returned: {text[:150]}...")
    return result


**Done when:** the OpenAI key is loaded, `uvx` is found and the GitHub token is loaded.

---
## Step 2: Connect to MCP Servers

**Goal:** **one client per server**, then test each connection by listing its tools and resources.

| | 🌐 Fetch | 🐙 GitHub |
|---|---|---|
| **Transport** | `stdio`: the client **starts** the server | `http`: the client **calls** the server |
| **Connection keys** | `command` + `args` | `url` + `headers` |
| **Secret** | None, it only reads public pages | A **token** in the headers |

### 2a. Fetch client (stdio)

The client starts the server exactly as if we typed this in a terminal (version pinned, as the course advises):

```
uvx mcp-server-fetch@2026.10.24
```

In [4]:
# Fetch client: local server, started by uvx
fetch_client = MultiServerMCPClient(
    {
        "fetch": {
            "transport": "stdio",
            "command": UVX,
            "args": ["mcp-server-fetch@2026.10.24"],
        }
    },
    tool_interceptors=[log_tool_calls],
)
print("Fetch client configured")

Fetch client configured


### 2b. GitHub client (HTTP + token)

GitHub hosts its official MCP server at `https://api.githubcopilot.com/mcp/`. We send **`headers`** with every request:

| Header | Value | Why |
|---|---|---|
| `Authorization` | `f"Bearer {GITHUB_TOKEN}"` | Our ID card. Without a valid token GitHub rejects the call (*401* or *400*) |
| `X-MCP-Toolsets` | `"repos,issues"` | Only load these groups of tools |
| `X-MCP-Readonly` | `"true"` | Only read tools: the agent cannot change anything on GitHub |

The npm package `@modelcontextprotocol/server-github` is **deprecated**, so we use GitHub's official server.

In [5]:
# GitHub client: remote server, token in the headers
github_client = MultiServerMCPClient(
    {
        "github": {
            "transport": "http",
            "url": "https://api.githubcopilot.com/mcp/",
            "headers": {
                "Authorization": f"Bearer {GITHUB_TOKEN}",
                "X-MCP-Toolsets": "repos,issues",
                "X-MCP-Readonly": "true",
            },
        }
    },
    tool_interceptors=[log_tool_calls],
)
print("GitHub client configured")

GitHub client configured


### 2c. Test both connections

For **each** client: count the tools with `get_tools()`, and try `get_resources()` inside a `try/except`. The first Fetch call can take ~20 s (`uvx` downloads the package).

In [6]:
clients = {"fetch": fetch_client, "github": github_client}

for name, client in clients.items():
    tools = await client.get_tools()
    print(f"{name}: {len(tools)} tools")
    try:
        resources = await client.get_resources()
        print(f"{name}: {len(resources)} resources")
    except Exception as error:
        print(f"{name}: no resources ({type(error).__name__})")

fetch: 1 tools
fetch: 0 resources
github: 19 tools
github: 1 resources


**Done when:** both clients return a tool count, and we know which server offers resources and which doesn't.


---
## Step 3: Load MCP Tools into LangChain

**Goal:** turn the MCP tools into LangChain tools and check they are accessible. `get_tools()` already converts every MCP tool into a LangChain tool.

In [7]:
fetch_tools = await fetch_client.get_tools()
github_tools = await github_client.get_tools()

print("Fetch tools: ", [t.name for t in fetch_tools])
print("GitHub tools:", [t.name for t in github_tools])

Fetch tools:  ['fetch']
GitHub tools: ['get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_release_by_tag', 'get_tag', 'issue_read', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_releases', 'list_repository_collaborators', 'list_tags', 'search_code', 'search_commits', 'search_issues', 'search_repositories']


**Done when:** we can see the name of every tool from both servers.

---
## Step 4: Create an Agent with MCP Tools

**Goal:** one agent per server that answers from its source, with a receipt. Each question is one the LLM **cannot** answer from memory, so it must call a tool.

In [8]:
# Fetch agent: answers from a documentation page
fetch_agent = create_agent(
    model=llm,
    tools=fetch_tools,
    system_prompt=(
        "You read web pages with the fetch tool. Always fetch the page before you answer, "
        "never guess, and name the URL you used."
    ),
)

question = "Read https://openai.github.io/openai-agents-python/. In one or two sentences: what does the OpenAI Agents SDK say it is for?"

fetch_result = await fetch_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:", fetch_result["messages"][-1].content)

[LOG] fetch -> fetch  {'url': 'https://openai.github.io/openai-agents-python/'}
[LOG]    returned: Contents of https://openai.github.io/openai-agents-python/:
<p>OpenAI Agents SDK</p>

Skip to content

# OpenAI Agents SDK

The OpenAI Agents SDK enab...
[LOG] fetch -> fetch  {'url': 'https://openai.github.io/openai-agents-python/', 'start_index': 5000}
[LOG]    returned: Contents of https://openai.github.io/openai-agents-python/:
esume state | Results |
| Build a low-latency voice agent with gpt-realtime-2.1 | Realtime...

Answer: The OpenAI Agents SDK is designed to enable the creation of agentic AI applications using a lightweight and user-friendly package, allowing developers to build complex workflows with minimal abstractions. It provides essential features such as agents equipped with instructions and tools, delegation capabilities, and validation mechanisms for inputs and outputs. 

You can find more information at [OpenAI Agents SDK](https://openai.github.io/openai-agents-python

In [9]:
# GitHub agent: answers from a repository
github_agent = create_agent(
    model=llm,
    tools=github_tools,
    system_prompt=(
        "You read public GitHub repositories with the GitHub tools. Never guess: every fact "
        "must come from a tool result. Name the repository and the file or tool you used."
    ),
)

question = (
    "For the GitHub repository langchain-ai/langgraph: which license does it use "
    "(read the LICENSE file), and what is its latest release?"
)

github_result = await github_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:", github_result["messages"][-1].content)

[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langgraph', 'path': 'LICENSE'}
[LOG] github -> get_latest_release  {'owner': 'langchain-ai', 'repo': 'langgraph'}
[LOG]    returned: {"tag_name":"1.2.18","target_commitish":"3521f08e21a7d89c28331191374bc85c11ecb6e2","name":"langgraph==1.2.18","body":"Changes since 1.2.17\n\n* releas...
[LOG]    returned: successfully downloaded text file (SHA: fc0602feecdd6748623c852ab534e1ca612673c7) type='resource' resource=TextResourceContents(uri=AnyUrl('repo://lan...

Answer: The repository **langchain-ai/langgraph** uses the **MIT License**. The license allows for free use, modification, and distribution of the software, provided that the copyright notice and permission notice are included in all copies or substantial portions of the software.

The latest release of the repository is **version 1.2.18**, which was published on **October 26, 2026**. You can find more details about this release [here](https://github.com/langchain-

---
## Step 5: Access MCP Resources

**Goal:** give an agent read-only **background knowledge** instead of tools.

| | Tools | Resources |
|---|---|---|
| **What** | Actions: *do* something | Read-only data: *know* something |
| **Who decides** | The **agent** calls them | **Our code** fetches them and hands them over |

Step 2c showed that Fetch offers **no** resources. So we use the **LangChain Docs server** from `mcp_introduction`, which offers one: the LangChain skill guide. It fits the RFP: the guide says **when to pick LangGraph**.

In [10]:
# LangChain Docs client: remote, no token
docs_client = MultiServerMCPClient(
    {"langchain-docs": {"transport": "http", "url": "https://docs.langchain.com/mcp"}}
)

resources = await docs_client.get_resources()
guide_text = resources[0].as_string()
print(f"{len(resources)} resource(s); the guide has {len(guide_text)} characters")

# Agent with NO tools: the guide text is its only source
guide_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "Answer ONLY from the guide below. If the guide does not say it, answer 'Not in the guide'.\n\n"
        f"GUIDE:\n{guide_text}"
    ),
)

question = "According to the guide, when should a team use LangGraph instead of LangChain's create_agent?"

guide_result = await guide_agent.ainvoke({"messages": [HumanMessage(content=question)]})
answer = guide_result["messages"][-1].content
print("\nAnswer:", answer)

# Evidence: a resource has no tool call, so we check key words from the answer in the resource text
for word in ["deterministic", "human-in-the-loop"]:
    print(f"'{word}': in answer = {word in answer.lower()}, in guide = {word in guide_text.lower()}")

1 resource(s); the guide has 11252 characters

Answer: A team should use LangGraph instead of LangChain's create_agent when they need to orchestrate complex workflows that mix deterministic logic with LLM-driven steps in a single graph. LangGraph provides low-level control for such scenarios, allowing for human-in-the-loop processes.
'deterministic': in answer = True, in guide = True
'human-in-the-loop': in answer = True, in guide = True


---
## Step 6: Build a Complete MCP-Enabled Agent

**Goal:** one agent that uses tools from **two servers**, through **two clients**, to answer the RFP question that needs **both**. An agent doesn't care where a tool comes from: it just gets a **list of tools**.

In [11]:
# Two clients -> one tool list (reuse the tools from Step 3)
all_tools = fetch_tools + github_tools

print(f"{len(fetch_tools)} Fetch tools + {len(github_tools)} GitHub tools = {len(all_tools)} tools")
print([t.name for t in all_tools])

1 Fetch tools + 19 GitHub tools = 20 tools
['fetch', 'get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_release_by_tag', 'get_tag', 'issue_read', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_releases', 'list_repository_collaborators', 'list_tags', 'search_code', 'search_commits', 'search_issues', 'search_repositories']


In [12]:
combined_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=(
        "You help an AI consultant shortlist agent frameworks for a client RFP. "
        "Use the GitHub tools for repository facts: the license from the LICENSE file, and the latest release. "
        "Use the fetch tool to read documentation pages. "
        "Never guess: every fact must come from a tool result, and name the repo, file or URL behind it. "
        "If the tools do not give a fact, write 'not found in the sources'."
    ),
)

question = (
    "We are shortlisting agent frameworks for a client RFP. "
    "For LangGraph (GitHub repo langchain-ai/langgraph, docs https://docs.langchain.com/oss/python/langgraph/overview) "
    "and the OpenAI Agents SDK (GitHub repo openai/openai-agents-python, docs https://openai.github.io/openai-agents-python/): "
    "give each one's license and latest release from GitHub, and one sentence from each docs page "
    "on what the framework is for. Answer as a small table with a source column."
)

combined_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:\n", combined_result["messages"][-1].content)

[LOG] github -> get_latest_release  {'owner': 'langchain-ai', 'repo': 'langgraph'}
[LOG] github -> get_latest_release  {'owner': 'openai', 'repo': 'openai-agents-python'}
[LOG] github -> get_file_contents  {'owner': 'langchain-ai', 'repo': 'langgraph', 'path': 'LICENSE'}
[LOG] github -> get_file_contents  {'owner': 'openai', 'repo': 'openai-agents-python', 'path': 'LICENSE'}
[LOG] fetch -> fetch  {'url': 'https://docs.langchain.com/oss/python/langgraph/overview'}
[LOG] fetch -> fetch  {'url': 'https://openai.github.io/openai-agents-python/'}
[LOG]    returned: Contents of https://openai.github.io/openai-agents-python/:
<p>OpenAI Agents SDK</p>

Skip to content

# OpenAI Agents SDK

The OpenAI Agents SDK enab...
[LOG]    returned: Contents of https://docs.langchain.com/oss/python/langgraph/overview:
<p>LangGraph overview - Docs by LangChain</p>

> ## Documentation Index
>
> Fetc...
[LOG]    returned: {"tag_name":"v0.24.0","target_commitish":"17d994bd3045dae71acda676a4ef4762ed8ecd94","na

### One question of our own: a failure probe

Pricing is **not** in our sources (repos and docs pages). A grounded agent must say so instead of inventing a number.

In [13]:
question = (
    "Does LangGraph or the OpenAI Agents SDK offer paid enterprise support, and what does it cost per year? "
    "Use only your tools."
)

own_result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})
print("\nAnswer:\n", own_result["messages"][-1].content)

[LOG] github -> search_repositories  {'query': 'LangGraph'}
[LOG] github -> search_repositories  {'query': 'OpenAI Agents SDK'}
[LOG]    returned: {"total_count":3595,"incomplete_results":false,"items":[{"id":1001787804,"name":"openai-cs-agents-demo","full_name":"openai/openai-cs-agents-demo","de...
[LOG]    returned: {"total_count":88933,"incomplete_results":false,"items":[{"id":676672661,"name":"langgraph","full_name":"langchain-ai/langgraph","description":"Build ...
[LOG] fetch -> fetch  {'url': 'https://github.com/langchain-ai/langgraph'}
[LOG] fetch -> fetch  {'url': 'https://github.com/openai/openai-cs-agents-demo'}
[LOG]    returned: Contents of https://github.com/langchain-ai/langgraph:
<p>GitHub - langchain-ai/langgraph: Build resilient agents. · GitHub</p>

Skip to content

## N...
[LOG]    returned: Contents of https://github.com/openai/openai-cs-agents-demo:
<p>GitHub - openai/openai-cs-agents-demo: Demo of a customer service use case implemented...
[LOG] github -> get_file

---
### Troubleshooting

| Problem | Fix |
|---|---|
| `uvx` not found / Fetch connection fails | Run `mcp_setup_mac.ipynb` (installs `uv`), then restart the kernel |
| GitHub `401 Unauthorized` or `400 Bad Request` | Token missing, wrong or expired: check `.env`, then restart the kernel |
| Fetch answers that robots.txt forbids the page | The site blocks automated fetching: use another page (e.g. the project README on GitHub) |
| You get a `coroutine` instead of a result | Add `await` before `get_tools()`, `get_resources()`, `ainvoke()` |
| `get_resources()` fails | That server offers no resources |